In [2]:
# Run cells selectively according to your needs.
from tqdm.auto import tqdm
import os
from pathlib import Path
import datasets
import multiprocess as mp
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")

DISKROOT = "/FS1"

hf_cache_dir = os.path.join(DISKROOT, "datasets/hf_datasets_cache")
dataset_path = os.path.join(DISKROOT, "datasets/OpenWebText")
os.makedirs(hf_cache_dir, exist_ok=True)
tokenized_dataset_path = dataset_path + '_tokenized'
chunked_tokenized_dataset_path = tokenized_dataset_path + '_chunked'

def tokenize(batch, tokenizer):
    return tokenizer(
        batch["text"], 
        truncation=True,        # Truncates texts longer than max_length
        max_length=1024         # Explicitly set to match your model's context
    )

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_DATASETS_CACHE"] = os.path.join(DISKROOT, "datasets/hf_datasets_cache")
num_proc = 7
try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass


In [ ]:
# 5. Import and load with explicit cache_dir override
from datasets import load_dataset
dataset = load_dataset(
    dataset_path, 
    split='train',
    cache_dir=hf_cache_dir  # Explicitly forces data writing to /FS1
)


In [ ]:
print(f'Starting tokenizing with num_proc={num_proc}')
token_ids = dataset.map(
    tokenize,
    batched=True,
    batch_size=1000,
    num_proc=num_proc,
    remove_columns=dataset.column_names,
    fn_kwargs={"tokenizer": tokenizer}  # <--- Serializes and sends tokenizer to workers
)

# 5. Persist final dataset
print('Tokenization completes. Saving to disk...')
token_ids.save_to_disk(tokenized_dataset_path)


In [ ]:
token_ids = load_from_disk(tokenized_dataset_path)

In [ ]:
eos_id= tokenizer.eos_token_id

def group_texts(examples, eos_id):
    # Append eos_id to each document before concatenating
    block_size = 1024
    concatenated_ids = []
    for seq in examples["input_ids"]:
        concatenated_ids.extend(seq)
        concatenated_ids.append(eos_id)

    total_length = len(concatenated_ids)
    total_length = (total_length // block_size) * block_size

    return {
        "input_ids": [
            concatenated_ids[i : i + block_size]
            for i in range(0, total_length, block_size)
        ]
    }

chunked_dataset = token_ids.map(
    group_texts,
    batched=True,
    num_proc=num_proc, # Now num_proc works here
    desc="Grouping texts",
    remove_columns=token_ids.column_names,
    fn_kwargs={"eos_id": eos_id}
)
chunked_dataset.save_to_disk(chunked_tokenized_dataset_path)

In [3]:
chunked_dataset = load_from_disk(chunked_tokenized_dataset_path)

Loading dataset from disk:   0%|          | 0/45 [00:00<?, ?it/s]

In [11]:
# list dataset splits
num_examples = len(chunked_dataset)
train_token_budget = 20 * 162 * 10**6
eval_token_budget = int(0.01 * train_token_budget)
train_examples_budget = train_token_budget // 1024
eval_examples_budget = eval_token_budget // 1024
print(f"Total examples: {num_examples}")
print(f"Train examples budget: {train_examples_budget}")
print(f"Eval examples budget: {eval_examples_budget}")

Total examples: 5458075
Train examples budget: 3164062
Eval examples budget: 31640


In [13]:
# split the dataset into train and eval based on the budget
train_dataset = chunked_dataset.select(range(train_examples_budget))
eval_dataset = chunked_dataset.select(range(train_examples_budget, train_examples_budget + eval_examples_budget))
remaining_examples = num_examples - (train_examples_budget + eval_examples_budget)
remaining_dataset = chunked_dataset.select(range(train_examples_budget + eval_examples_budget, num_examples))

In [14]:
print(f"Train dataset size: {len(train_dataset)}")
print(f"Eval dataset size: {len(eval_dataset)}")
print(f"Remaining dataset size: {len(remaining_dataset)}")
# save all three datasets to disk
train_dataset.save_to_disk(os.path.join(DISKROOT, "datasets/openwebtext_tokenized_chunked_train"))
eval_dataset.save_to_disk(os.path.join(DISKROOT, "datasets/openwebtext_tokenized_chunked_eval"))
remaining_dataset.save_to_disk(os.path.join(DISKROOT, "datasets/openwebtext_tokenized_chunked_remaining"))

Train dataset size: 3164062
Eval dataset size: 31640
Remaining dataset size: 2262373


Saving the dataset (0/26 shards):   0%|          | 0/3164062 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/31640 [00:00<?, ? examples/s]

Saving the dataset (0/19 shards):   0%|          | 0/2262373 [00:00<?, ? examples/s]